# Language Agent Tree Search (LATS) | Advanced Planning & Search

In [1]:
import math
import re
from dataclasses import dataclass, field
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
@dataclass
class TreeNode:
    state: str  # partial solution steps so far
    parent: Optional['TreeNode'] = None
    children: List['TreeNode'] = field(default_factory=list)
    visits: int = 0
    value: float = 0.0

    def ucb1(self, c: float = 1.41) -> float:
        if self.visits == 0:
            return float('inf')
        return (self.value / self.visits) + c * math.sqrt(math.log(self.parent.visits) / self.visits)

def select(node: TreeNode) -> TreeNode:
    """Walk down the tree picking the child with highest UCB1."""
    while node.children:
        node = max(node.children, key=lambda n: n.ucb1())
    return node

def expand(node: TreeNode, problem: str) -> None:
    """LLM proposes 3 candidate next steps from the current partial solution."""
    resp = model.invoke(
        f"Problem: {problem}\nPartial solution so far:\n{node.state}\n\n"
        "Propose exactly 3 distinct next steps (one line each, numbered 1-3). "
        "Each step should advance the solution."
    )
    for line in resp.content.strip().splitlines():
        line = line.strip()
        if line and line[0].isdigit():
            step = re.sub(r"^\d+[\.)\s]", "", line)
            child_state = f"{node.state}\n- {step}"
            node.children.append(TreeNode(state=child_state, parent=node))

def simulate(node: TreeNode, problem: str) -> float:
    """LLM scores how promising this partial solution is (0.0 to 1.0)."""
    resp = model.invoke(
        f"Problem: {problem}\nPartial solution:\n{node.state}\n\n"
        "Rate how promising this partial solution is on a scale of 0.0 to 1.0. "
        "Reply with ONLY a decimal number."
    )
    try:
        return max(0.0, min(1.0, float(re.search(r"[\d.]+", resp.content).group())))
    except (ValueError, AttributeError):
        return 0.5

def backpropagate(node: TreeNode, value: float) -> None:
    while node:
        node.visits += 1
        node.value += value
        node = node.parent

def best_path(root: TreeNode) -> str:
    """Follow highest-value children from root to leaf."""
    node, path = root, root.state
    while node.children:
        node = max(node.children, key=lambda n: n.value / max(n.visits, 1))
        path = node.state
    return path

In [5]:
# --- Run LATS on a math problem ---
problem = "Find two numbers that multiply to 60 and add to 17."
root = TreeNode(state="Start: find x and y where x*y=60 and x+y=17")

for i in range(12):
    leaf = select(root)
    if leaf.visits > 0 and not leaf.children:
        expand(leaf, problem)
        leaf = leaf.children[0] if leaf.children else leaf
    score = simulate(leaf, problem)
    backpropagate(leaf, score)
    print(f"Iteration {i+1}: score={score:.2f}, tree size={root.visits}")

print(f"\nBest path found:\n{best_path(root)}")

Iteration 1: score=1.00, tree size=1
Iteration 2: score=0.90, tree size=2
Iteration 3: score=1.00, tree size=3
Iteration 4: score=1.00, tree size=4
Iteration 5: score=0.90, tree size=5
Iteration 6: score=1.00, tree size=6
Iteration 7: score=0.90, tree size=7
Iteration 8: score=1.00, tree size=8
Iteration 9: score=1.00, tree size=9
Iteration 10: score=1.00, tree size=10
Iteration 11: score=1.00, tree size=11
Iteration 12: score=0.90, tree size=12

Best path found:
Start: find x and y where x*y=60 and x+y=17
-  Simplify and solve the quadratic equation x^2 - 17x + 60 = 0 to find the values of x.
-  Calculate the discriminant of the quadratic equation, \(\Delta = b^2 - 4ac = 17^2 - 4 \cdot 1 \cdot 60\).


In [6]:
# --- Greedy comparison: always pick first expansion, no UCB1 ---
greedy_root = TreeNode(state="Start: find x and y where x*y=60 and x+y=17")
for _ in range(12):
    expand(greedy_root, problem)
    greedy_root = greedy_root.children[0] if greedy_root.children else greedy_root
greedy_score = simulate(greedy_root, problem)
mcts_score = root.value / max(root.visits, 1)
print(f"\nMCTS best: {mcts_score:.2f} vs Greedy: {greedy_score:.2f}")


MCTS best: 0.97 vs Greedy: 1.00
